# Đếm rep tập tạ đơn (dumbbell curl) bằng YOLO-Pose

Đếm số lần cuốn tạ và tách các đoạn **đang tập / đang nghỉ** từ video, dùng
`ultralytics` YOLO11-pose thay cho MediaPipe.

## Cách chạy trên Kaggle
1. Settings → **Accelerator: GPU T4 x2** (chạy CPU vẫn được nhưng chậm hơn ~10×).
2. Settings → **Internet: On** — bắt buộc, vì notebook tải video từ Google Drive
   bằng `gdown` và tải trọng số YOLO.
3. Run All. Video được tải tự động, không cần tạo Dataset.

## Kiến trúc 2 lượt (2-pass)

| Lượt | Việc | Chi phí |
|---|---|---|
| 1 | Chạy YOLO-pose một lần, lưu keypoint ra `.npz` | chậm (phần nặng nhất) |
| 2 | Tính tín hiệu → đếm rep → tách set → vẽ video | vài giây |

Tách như vậy để **chỉnh ngưỡng mà không phải chạy lại YOLO** — chạy lại ô phân tích là đủ.
Notebook push-up cũ gộp tất cả vào một vòng lặp, nên mỗi lần đổi ngưỡng phải suy luận
lại toàn bộ video.

## Tín hiệu dùng để đếm

Video này quay rất gần, **đầu bị cắt khỏi khung** và vai thường bị che, nên keypoint vai
có độ tin cậy thấp và dao động mạnh (đo được 0.09–0.93 trên cùng một video). Vì vậy
**không** lấy góc khuỷu `vai–khuỷu–cổ tay` làm tín hiệu chính. Thay vào đó dùng
**độ nghiêng cẳng tay**:

$$\mathrm{fore} \;=\; \frac{y_{\text{khuỷu}} - y_{\text{cổ tay}}}{\lVert \vec{p}_{\text{khuỷu}} - \vec{p}_{\text{cổ tay}} \rVert} \;\in\; [-1, +1]$$

- Chỉ cần **khuỷu + cổ tay** — hai keypoint tin cậy nhất (~0.98).
- Tự chuẩn hoá theo chiều dài cẳng tay ⇒ không phụ thuộc độ phân giải, khoảng cách
  camera hay dáng người. (Notebook push-up cũ so ngưỡng pixel cứng `std < 100`, đổi độ
  phân giải là hỏng.)
- Duỗi thẳng tay xuống → `fore ≈ −1`; cuốn lên ngang ngực → `fore ≈ +0.7`.

Góc khuỷu vẫn được tính song song để đối chiếu và hiển thị.

In [ ]:
!pip install -q ultralytics
!pip install -q --upgrade gdown

In [ ]:
import os, re, time, math
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from ultralytics import YOLO
import torch

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Torch', torch.__version__, '| device =', DEVICE)
print('OpenCV', cv2.__version__)

## Cấu hình

Các ngưỡng dưới đây được dò từ chính video `IMG_2425.MOV` (13.7 phút, 1080×1920, 30 fps):
trích keypoint toàn bộ video rồi soi phân bố tín hiệu. Nếu dùng video khác, chạy ô
**"Dò ngưỡng tự động"** ở gần cuối để lấy bộ ngưỡng mới.

In [ ]:
# ============================ CẤU HÌNH ============================
# --- Nguồn video: tải từ Google Drive bằng gdown ---
GDRIVE_URL = 'https://drive.google.com/file/d/1lHKOfD07OlZj9ZkPLHduob5-cfhqLM0B/view'
VIDEO_NAME = 'IMG_2425.MOV'

OUT_DIR    = '/kaggle/working'
# Tải vào /kaggle/temp: thư mục này KHÔNG bị tính vào dung lượng output của notebook
# (video ~925 MB, để trong /kaggle/working là chạm trần 20 GB khi commit).
DL_DIR     = '/kaggle/temp' if os.path.isdir('/kaggle') else OUT_DIR
VIDEO_PATH = f'{DL_DIR}/{VIDEO_NAME}'
# Nếu đã upload video thành Kaggle Dataset thì trỏ thẳng vào đó và bỏ qua ô tải:
# VIDEO_PATH = '/kaggle/input/ta-don/IMG_2425.MOV'

KPTS_NPZ   = f'{OUT_DIR}/kpts.npz'                 # cache keypoint của lượt 1
OUT_VIDEO  = f'{OUT_DIR}/curl_output.mp4'

# --- Lượt 1: trích keypoint ---
MODEL_NAME = 'yolo11s-pose.pt'  # 's' thay vì 'n': độ tin cậy vai 0.09 -> 0.84 trên video này
IMGSZ      = 640
BOX_CONF   = 0.25    # ngưỡng tin cậy box người
KP_CONF    = 0.50    # keypoint yếu hơn mức này coi như không có (NaN), không dùng bừa
STRIDE     = 2       # suy luận mỗi N frame; 2 -> 15 Hz, thừa sức bắt rep dài ~2 s
BATCH      = 16      # gộp frame theo lô cho GPU

# --- Lượt 2: đếm rep ---
SMOOTH_WIN_S = 0.25   # cửa sổ làm mượt Savitzky-Golay (giây). Nhịp cuốn đo được ~1.4 s
MAX_GAP_S    = 0.50   # mất pose ngắn hơn mức này thì nội suy bù

REP_HI         = +0.15  # đi lên vượt ngưỡng này -> tính 1 rep
REP_LO         = -0.55  # tụt xuống dưới ngưỡng này -> về trạng thái đáy
REP_MIN_AMP    = 0.60   # biên độ đáy->đỉnh tối thiểu, loại rung lắc nhỏ
REP_MIN_PERIOD = 0.50   # khoảng cách tối thiểu giữa 2 rep (giây), chống đếm kép
MERGE_WIN_S    = 0.40   # 2 tay lên cách nhau dưới mức này -> tính LÀ MỘT rep

# --- Lượt 2: tách set tập / nghỉ ---
SET_MAX_GAP_S     = 6.0   # 2 rep cách nhau lâu hơn mức này -> thuộc 2 set khác nhau
SET_PAD_S         = 1.5   # nới đầu/cuối set để bao trọn nhịp nâng đầu và cuối
SET_MIN_REPS      = 3     # set ít rep hơn mức này bị loại (động tác lẻ, gãi, chỉnh tạ)
SET_MAX_CADENCE_S = 3.0   # nhịp trung vị chậm hơn mức này thì không coi là set thật

# --- Lượt 2: vẽ video ---
RENDER_VIDEO = True
BLUR_FACE    = True   # làm mờ mặt nếu mặt lọt khung (video này đầu bị cắt nên thường vô hiệu)
DRAW_PLOT    = True   # vẽ đồ thị tín hiệu chạy trên frame (bằng cv2, không dùng matplotlib)
# ==================================================================

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(DL_DIR, exist_ok=True)
print('Video sẽ nằm ở:', VIDEO_PATH)

## Tải video từ Google Drive

Cần bật **Internet: On** trong Settings, và link Drive phải ở chế độ chia sẻ
*Anyone with the link*. Ô này bỏ qua nếu file đã có sẵn, nên chạy lại notebook
không tải lại từ đầu.

In [ ]:
import gdown

def drive_id(url):
    """Tách file id khỏi link Drive (chấp nhận cả dạng /file/d/<id>/view lẫn ?id=<id>)."""
    m = (re.search(r'/d/([A-Za-z0-9_-]{20,})', url)
         or re.search(r'[?&]id=([A-Za-z0-9_-]{20,})', url))
    assert m, f'Không tách được file id từ link: {url}'
    return m.group(1)


def fetch_video(url, dest, min_mb=10):
    if os.path.exists(dest) and os.path.getsize(dest) > min_mb * 1e6:
        print(f'Đã có sẵn: {dest} ({os.path.getsize(dest)/1e6:.0f} MB) — bỏ qua tải.')
    else:
        print('Đang tải từ Google Drive...')
        # Truyền id= thay vì url=: gdown >= 5 đã bỏ tham số fuzzy, còn id= thì
        # bản nào cũng nhận, nên notebook không phụ thuộc version gdown của Kaggle.
        gdown.download(id=drive_id(url), output=dest, quiet=False)

    assert os.path.exists(dest), (
        f'Tải thất bại. Kiểm tra: (1) Internet đã bật trong Settings? '
        f'(2) Link Drive đã chia sẻ "Anyone with the link"?')

    mb = os.path.getsize(dest) / 1e6
    # gdown gặp trang chặn của Drive sẽ lưu chính trang HTML đó thành file -> vài KB
    assert mb > min_mb, (
        f'File chỉ {mb:.2f} MB — gần như chắc chắn là trang lỗi HTML chứ không phải video. '
        f'Thường do quota tải của Drive, hoặc link chưa mở công khai.')

    cap = cv2.VideoCapture(dest)
    ok  = cap.isOpened() and cap.read()[0]
    cap.release()
    assert ok, f'OpenCV không đọc được {dest} — file hỏng hoặc codec không hỗ trợ.'
    print(f'OK: {dest} ({mb:.0f} MB)')

fetch_video(GDRIVE_URL, VIDEO_PATH)

## Hàm dùng chung

Chỉ số keypoint theo chuẩn COCO-17 mà YOLO-pose xuất ra.

In [ ]:
# COCO-17
NOSE, L_EYE, R_EYE, L_EAR, R_EAR = 0, 1, 2, 3, 4
L_SHO, R_SHO, L_ELB, R_ELB, L_WRI, R_WRI = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KNE, R_KNE, L_ANK, R_ANK = 11, 12, 13, 14, 15, 16
FACE_KPS = [NOSE, L_EYE, R_EYE, L_EAR, R_EAR]

SKELETON = [(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
            (11,13),(13,15),(12,14),(14,16)]

SIDES = {'L': (L_SHO, L_ELB, L_WRI), 'R': (R_SHO, R_ELB, R_WRI)}


def angle_at(a, b, c):
    """Góc tại đỉnh b (độ). a/b/c dạng (N,2). NaN nếu cạnh suy biến."""
    v1, v2 = a - b, c - b
    n1 = np.linalg.norm(v1, axis=-1)
    n2 = np.linalg.norm(v2, axis=-1)
    cos = np.sum(v1 * v2, -1) / np.clip(n1 * n2, 1e-6, None)
    out = np.degrees(np.arccos(np.clip(cos, -1, 1)))
    out[(n1 < 1e-3) | (n2 < 1e-3)] = np.nan
    return out


def build_signals(kpts, kp_conf=KP_CONF):
    """(N,17,3) -> dict tín hiệu 1 chiều. Keypoint yếu thành NaN thay vì dùng bừa."""
    xy, kc = kpts[..., :2].astype(np.float64), kpts[..., 2]
    out = {}
    for s, (i_sho, i_elb, i_wri) in SIDES.items():
        sho, elb, wri = xy[:, i_sho], xy[:, i_elb], xy[:, i_wri]

        # (a) Góc khuỷu — cần cả vai, mà vai là keypoint yếu nhất ở video này
        ang = angle_at(sho, elb, wri)
        ang[np.minimum.reduce([kc[:, i_sho], kc[:, i_elb], kc[:, i_wri]]) < kp_conf] = np.nan

        # (b) Độ nghiêng cẳng tay — chỉ cần khuỷu + cổ tay. ĐÂY LÀ TÍN HIỆU CHÍNH.
        #     Trục y của ảnh hướng xuống, nên (y_khuỷu - y_cổtay) > 0 <=> cổ tay CAO hơn khuỷu.
        fv   = elb - wri
        flen = np.linalg.norm(fv, axis=-1)
        fore = fv[:, 1] / np.clip(flen, 1e-6, None)
        fore[(np.minimum(kc[:, i_elb], kc[:, i_wri]) < kp_conf) | (flen < 1e-3)] = np.nan

        out[f'ang{s}']  = ang
        out[f'fore{s}'] = fore
    return out


def interp_short_gaps(x, max_gap):
    """Nội suy tuyến tính các khoảng NaN NGẮN. Khoảng dài giữ NaN (mất pose thật)."""
    x  = np.asarray(x, float).copy()
    ok = np.isfinite(x)
    if ok.sum() < 2:
        return x
    i = np.arange(len(x))
    filled = np.interp(i, i[ok], x[ok])
    run_id = np.cumsum(ok)                    # đánh số từng cụm NaN liên tiếp
    for g in np.unique(run_id[~ok]):
        m = (~ok) & (run_id == g)
        if m.sum() > max_gap:                 # cụm quá dài -> trả lại NaN
            filled[m] = np.nan
    filled[:np.argmax(ok)] = np.nan           # không ngoại suy hai đầu
    filled[len(x) - np.argmax(ok[::-1]):] = np.nan
    return filled


def smooth(x, win):
    """Savitzky-Golay bỏ qua NaN: lọc riêng từng đoạn liên tục để NaN không lan ra."""
    win = max(5, int(win) | 1)
    y   = np.full(len(x), np.nan, dtype=float)
    ok  = np.isfinite(x)
    if ok.sum() < 5:
        return np.asarray(x, float).copy()
    edges  = np.flatnonzero(np.diff(ok.astype(int)))
    bounds = np.concatenate([[0], edges + 1, [len(x)]])
    for a, b in zip(bounds[:-1], bounds[1:]):
        if not ok[a]:
            continue
        y[a:b] = savgol_filter(x[a:b], win, 2) if (b - a) >= win else x[a:b]
    return y

---
## Lượt 1 — Trích keypoint

Chạy YOLO-pose một lần trên toàn video, lưu ra `kpts.npz`. Khi có nhiều người trong
khung thì lấy **box lớn nhất** (người gần camera nhất) chứ không lấy box có confidence
cao nhất — trên video này đã gặp frame có 2 box, và box lớn nhất mới là chủ thể.

Ô này tự bỏ qua nếu `kpts.npz` đã tồn tại; xoá file đó đi nếu muốn trích lại.

In [ ]:
def extract_keypoints(video_path, model, stride=STRIDE, batch=BATCH,
                      imgsz=IMGSZ, conf=BOX_CONF, device=DEVICE):
    """Trả về (idx, kpts (N,17,3), bconf, meta). Frame không thấy người -> NaN."""
    cap = cv2.VideoCapture(video_path)
    assert cap.isOpened(), f'Không mở được video: {video_path}'
    meta = dict(fps    = cap.get(cv2.CAP_PROP_FPS),
                width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
                height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
                total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    print('{width}x{height}  {fps:.2f} fps  {total} frame  '
          '({dur:.1f}s)'.format(dur=meta['total'] / max(meta['fps'], 1e-9), **meta))

    idx, kpts, bconf = [], [], []
    buf, buf_i = [], []

    def flush():
        if not buf:
            return
        for bi, r in zip(buf_i, model.predict(buf, imgsz=imgsz, conf=conf,
                                              device=device, verbose=False)):
            n = 0 if r.boxes is None else len(r.boxes)
            if n:
                b    = r.boxes.xyxy.cpu().numpy()
                area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
                j    = int(np.argmax(area))          # người to nhất = chủ thể
                xy   = r.keypoints.xy.cpu().numpy()[j]        # (17,2)
                kc   = r.keypoints.conf.cpu().numpy()[j]      # (17,)
                kpts.append(np.concatenate([xy, kc[:, None]], 1))
                bconf.append(float(r.boxes.conf.cpu().numpy()[j]))
            else:
                kpts.append(np.full((17, 3), np.nan, np.float32))
                bconf.append(0.0)
            idx.append(bi)
        buf.clear(); buf_i.clear()

    i, t0 = 0, time.time()
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if i % stride == 0:
            buf.append(frame); buf_i.append(i)
            if len(buf) == batch:
                flush()
                if len(idx) % (batch * 20) == 0:
                    done = i / max(meta['total'], 1)
                    el   = time.time() - t0
                    print(f'  {i}/{meta["total"]} ({done*100:5.1f}%)  '
                          f'{el:5.0f}s  còn ~{el/max(done,1e-9)-el:.0f}s', flush=True)
        i += 1
    flush()
    cap.release()
    print(f'Xong {len(idx)} mẫu trong {time.time()-t0:.0f}s')
    return np.array(idx), np.array(kpts, np.float32), np.array(bconf, np.float32), meta


def cache_is_valid(path, video, stride, model_name):
    """Cache chỉ dùng lại được khi khớp CẢ video, stride và model.

    Khoá cache theo mỗi tên file là bẫy: đổi video mà quên xoá .npz thì notebook
    lặng lẽ phân tích keypoint của video cũ rồi vỡ ở bước vẽ vì lệch số frame.
    """
    if not os.path.exists(path):
        return False
    try:
        d = np.load(path)
        same = (str(d['src']) == str(video) and int(d['stride']) == int(stride)
                and str(d['model']) == str(model_name))
    except Exception as e:
        print('Cache hỏng, trích lại:', e)
        return False
    if not same:
        print(f'Cache của video/tham số khác -> trích lại '
              f'(cache: {str(d["src"])}, stride={int(d["stride"])}, model={str(d["model"])})')
    return same


if cache_is_valid(KPTS_NPZ, VIDEO_PATH, STRIDE, MODEL_NAME):
    print('Dùng lại cache:', KPTS_NPZ)
else:
    model = YOLO(MODEL_NAME)
    idx, kpts, bconf, meta = extract_keypoints(VIDEO_PATH, model)
    np.savez_compressed(KPTS_NPZ, idx=idx, kpts=kpts, bconf=bconf,
                        stride=STRIDE, src=VIDEO_PATH, model=MODEL_NAME, **meta)

_d    = np.load(KPTS_NPZ)
idx   = _d['idx']
kpts  = _d['kpts']
bconf = _d['bconf']
FPS   = float(_d['fps'])
W, H  = int(_d['width']), int(_d['height'])
TOTAL = int(_d['total'])
t     = idx / FPS                      # mốc thời gian của từng mẫu (giây)
SR    = FPS / float(_d['stride'])      # tần số lấy mẫu của tín hiệu (Hz)

print(f'{len(idx)} mẫu @ {SR:.1f} Hz | thấy người ở {np.isfinite(kpts[:,L_ELB,0]).mean()*100:.1f}% mẫu')

---
## Lượt 2a — Dựng tín hiệu

Nội suy các khoảng mất pose ngắn rồi làm mượt. Giữ **hai tay tách riêng** — bước đếm
ở dưới sẽ tự xử lý chuyện cuốn đồng thời hay xen kẽ.

In [ ]:
S = build_signals(kpts)

max_gap_n = max(1, int(round(MAX_GAP_S * SR)))
win_n     = max(5, int(round(SMOOTH_WIN_S * SR)) | 1)
print(f'làm mượt {win_n} mẫu ({win_n/SR:.2f}s), bù khoảng trống tối đa {max_gap_n} mẫu')

for k in ['foreL', 'foreR', 'angL', 'angR']:
    S[k + '_s'] = smooth(interp_short_gaps(S[k], max_gap_n), win_n)

SIGNALS = {'Tay trái': S['foreL_s'], 'Tay phải': S['foreR_s']}

# chỉ dùng để hiển thị, không dùng để đếm
with np.errstate(invalid='ignore'):
    fore_both = np.nanmean(np.vstack([S['foreL_s'], S['foreR_s']]), axis=0)

for k in ['foreL', 'foreR', 'angL', 'angR']:
    v = S[k]
    print(f'{k:6s} hợp lệ {np.isfinite(v).mean()*100:5.1f}%  '
          f'p05={np.nanpercentile(v,5):7.2f}  med={np.nanmedian(v):7.2f}  '
          f'p95={np.nanpercentile(v,95):7.2f}')

---
## Lượt 2b — Đếm rep

Dùng **trigger Schmitt** (ngưỡng kép có trễ) thay cho FSM 3 trạng thái của notebook
push-up cũ:

- Đang ở `đáy`, tín hiệu vượt `REP_HI` → **+1 rep**, chuyển sang `đỉnh`.
- Đang ở `đỉnh`, tín hiệu tụt dưới `REP_LO` → về `đáy`.

Hai chốt chặn nhiễu:

- **Biên độ**: chênh lệch đáy→đỉnh phải ≥ `REP_MIN_AMP`, loại rung tay khi đứng nghỉ.
- **Chu kỳ**: hai rep phải cách nhau ≥ `REP_MIN_PERIOD` giây.

Khác biệt so với notebook cũ: ở đó điều kiện đếm dùng `velocity` lấy từ 2 mẫu cuối
nhưng `avg_angle` lại là trung bình 15 mẫu — hai đại lượng lệch pha nhau, nên phải
vừa đúng ngưỡng góc vừa đúng dấu vận tốc mới đếm, dễ bỏ sót rep. Ở đây chỉ dựa vào
mức tín hiệu đã làm mượt, không cần vận tốc.

### Ghép hai tay

Mỗi tay được đếm **độc lập**, sau đó hai rep của hai tay cách nhau dưới `MERGE_WIN_S`
giây được gộp làm một. Cách này tự đúng cho cả ba kiểu tập, không cần khai báo trước:

| Kiểu tập | Kết quả |
|---|---|
| Cuốn 2 tay đồng thời | 2 rep trùng thời điểm → gộp thành 1 |
| Cuốn xen kẽ | lệch nhau nửa chu kỳ (~0.7 s) > `MERGE_WIN_S` → đếm riêng |
| Chỉ một tay | tay kia im, không sinh rep → đếm đúng tay đang tập |

Lấy **trung bình hai tay rồi mới đếm** thì hỏng ở kiểu thứ ba: trên video này, đoạn
780–785 s chỉ tay trái cuốn, tín hiệu trung bình chỉ lên tới ≈ +0.05 nên không vượt
`REP_HI` và mất trắng các rep đó.

In [ ]:
def detect_reps(t, s, arm, hi=REP_HI, lo=REP_LO,
                min_amp=REP_MIN_AMP, min_period=REP_MIN_PERIOD):
    """Trigger Schmitt trên tín hiệu đã làm mượt -> danh sách rep của MỘT tay."""
    reps, state = [], None
    trough, trough_t = np.inf, np.nan
    last_rep_t = -np.inf

    for i in range(len(s)):
        v = s[i]
        if not np.isfinite(v):
            state = None                      # mất pose -> chờ khoá lại trạng thái
            continue
        if state is None:
            state = 'up' if v > hi else 'down'
            trough, trough_t = v, t[i]
            continue

        if state == 'down':
            if v < trough:
                trough, trough_t = v, t[i]
            if v > hi:
                amp = v - trough
                if amp >= min_amp and (t[i] - last_rep_t) >= min_period:
                    reps.append(dict(t_top=t[i], t_bottom=trough_t,
                                     bottom=trough, top=v, amp=amp, arm=arm))
                    last_rep_t = t[i]
                state = 'up'
        else:                                 # state == 'up'
            if v < lo:
                state, trough, trough_t = 'down', v, t[i]
    return reps


def merge_arms(reps_l, reps_r, win=MERGE_WIN_S):
    """Gộp rep của hai tay xảy ra gần nhau thành một (cuốn đồng thời)."""
    out = []
    for r in sorted(reps_l + reps_r, key=lambda r: r['t_top']):
        prev = out[-1] if out else None
        if prev and prev['arm'] != r['arm'] and r['t_top'] - prev['t_top'] <= win:
            prev['arm']      = 'LR'
            prev['t_top']    = (prev['t_top'] + r['t_top']) / 2
            prev['t_bottom'] = min(prev['t_bottom'], r['t_bottom'])
            prev['amp']      = max(prev['amp'], r['amp'])
            prev['top']      = max(prev['top'], r['top'])
        else:
            out.append(dict(r))
    return out


reps_L = detect_reps(t, S['foreL_s'], 'L')
reps_R = detect_reps(t, S['foreR_s'], 'R')
all_reps = merge_arms(reps_L, reps_R)
TOTAL_REPS = len(all_reps)

n_both = sum(1 for r in all_reps if r['arm'] == 'LR')
print(f'tay trái {len(reps_L)} + tay phải {len(reps_R)}  ->  {TOTAL_REPS} rep sau khi gộp')
print(f'  {n_both} rep hai tay đồng thời, {TOTAL_REPS - n_both} rep một tay')

---
## Lượt 2c — Tách đoạn TẬP / NGHỈ

Notebook push-up cũ nhận diện nghỉ bằng độ lệch chuẩn vị trí vai **hoặc** vai nằm cao
trong khung (`low_movement or high_position`), với ngưỡng pixel cứng. Điều đó khiến
người tập chậm và đều bị coi là đang nghỉ và **mất rep**.

Ở đây đoạn tập được suy ra **trực tiếp từ các rep đã đếm được**: rep là bằng chứng
đang tập, nên không thể mâu thuẫn với bộ đếm. Các rep liên tiếp cách nhau dưới
`SET_MAX_GAP_S` giây được gom thành một **set**; ngoài set là **nghỉ**.

Hai bộ lọc để không biến động tác lẻ thành "set":

- **Số rep**: dưới `SET_MIN_REPS` thì bỏ (gãi lưng, nhấc tạ lên chỉnh tay…).
- **Nhịp**: set thật có tiết tấu đều. Nhịp trung vị chậm hơn `SET_MAX_CADENCE_S`
  thì loại. Trên video này, lọc nhịp cắt 17 cụm ứng viên xuống còn 9 set thật.

In [ ]:
def group_sets(reps, max_gap=SET_MAX_GAP_S, pad=SET_PAD_S,
               min_reps=SET_MIN_REPS, max_cadence=SET_MAX_CADENCE_S):
    """Gom rep liên tiếp thành set. Trả list dict(start, end, n_reps, reps, dur)."""
    if not reps:
        return []
    groups, cur = [], [reps[0]]
    for r in reps[1:]:
        if r['t_top'] - cur[-1]['t_top'] <= max_gap:
            cur.append(r)
        else:
            groups.append(cur)
            cur = [r]            # list MỚI — đừng dùng cur.clear(), mọi group sẽ trỏ chung
    groups.append(cur)

    sets = []
    for g in groups:
        if len(g) < min_reps:
            continue
        gaps = np.diff([r['t_top'] for r in g])
        if len(gaps) and np.median(gaps) > max_cadence:   # không có tiết tấu -> không phải set
            continue
        sets.append(dict(start=max(0.0, min(r['t_bottom'] for r in g) - pad),
                         end=max(r['t_top'] for r in g) + pad,
                         n_reps=len(g), reps=list(g)))

    # nới biên bằng pad có thể làm 2 set liền nhau chồng lên nhau -> nhập lại làm một
    out = []
    for s in sets:
        if out and s['start'] <= out[-1]['end']:
            out[-1]['end']    = max(out[-1]['end'], s['end'])
            out[-1]['n_reps'] += s['n_reps']
            out[-1]['reps']   += s['reps']
        else:
            out.append(s)
    for s in out:
        s['dur'] = s['end'] - s['start']
    return out


SETS = group_sets(all_reps)

# Nhãn TẬP/NGHỈ cho từng mẫu
is_working = np.zeros(len(t), bool)
set_id     = np.full(len(t), -1, int)
for k, st in enumerate(SETS):
    m = (t >= st['start']) & (t <= st['end'])
    is_working[m] = True
    set_id[m]     = k

# Số rep tích luỹ tại mỗi mẫu (để hiển thị trên video)
rep_times   = np.array([r['t_top'] for r in all_reps]) if all_reps else np.zeros(0)
cum_reps    = np.searchsorted(rep_times, t, side='right')
# Số rep tích luỹ TRONG set hiện tại
reps_in_set = np.zeros(len(t), int)
for k, st in enumerate(SETS):
    m = set_id == k
    tops = np.array([r['t_top'] for r in st['reps']])
    reps_in_set[m] = np.searchsorted(tops, t[m], side='right')

work_s = is_working.sum() / SR
rest_s = (~is_working).sum() / SR
print(f'{len(SETS)} set | tập {work_s:.0f}s ({work_s/(work_s+rest_s)*100:.1f}%) | '
      f'nghỉ {rest_s:.0f}s')

In [ ]:
rows = []
for k, st in enumerate(SETS, 1):
    gaps = np.diff([r['t_top'] for r in st['reps']])
    rows.append({
        'Set': k,
        'Bắt đầu': time.strftime('%M:%S', time.gmtime(st['start'])),
        'Kết thúc': time.strftime('%M:%S', time.gmtime(st['end'])),
        'Thời lượng (s)': round(st['dur'], 1),
        'Số rep': st['n_reps'],
        'Nhịp TB (s/rep)': round(float(gaps.mean()), 2) if len(gaps) else np.nan,
        'Biên độ TB': round(float(np.mean([r['amp'] for r in st['reps']])), 2),
    })
df_sets = pd.DataFrame(rows)

# Thời gian nghỉ giữa các set
if len(SETS) > 1:
    df_sets['Nghỉ trước set (s)'] = [np.nan] + [
        round(SETS[i]['start'] - SETS[i-1]['end'], 1) for i in range(1, len(SETS))]

df_sets.to_csv(f'{OUT_DIR}/sets.csv', index=False)
pd.DataFrame(all_reps).drop(columns=['reps'], errors='ignore') \
    .to_csv(f'{OUT_DIR}/reps.csv', index=False)
display(df_sets)
print(f'\nTỔNG: {TOTAL_REPS} rep trong {len(SETS)} set')

---
## Biểu đồ kiểm tra

Đây là chỗ để **mắt thường xác nhận** ngưỡng đặt đúng: mỗi chấm đỏ phải rơi đúng đỉnh
một nhịp cuốn, và mỗi dải xanh phải trùng một loạt nhịp liên tiếp.

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(20, 10), sharex=True,
                       gridspec_kw=dict(height_ratios=[3, 2, 1]))

# (1) Tín hiệu đếm + rep + set
a = ax[0]
for st in SETS:
    a.axvspan(st['start'], st['end'], color='tab:green', alpha=.13, lw=0)
for name, s in SIGNALS.items():
    a.plot(t, s, lw=.9, label=name)
if all_reps:
    a.plot([r['t_top'] for r in all_reps], [r['top'] for r in all_reps],
           'v', color='crimson', ms=5, label=f'rep ({TOTAL_REPS})')
a.axhline(REP_HI, color='crimson', ls='--', lw=.8)
a.axhline(REP_LO, color='steelblue', ls='--', lw=.8)
a.text(t[-1], REP_HI, ' REP_HI', va='bottom', ha='right', color='crimson', fontsize=8)
a.text(t[-1], REP_LO, ' REP_LO', va='top', ha='right', color='steelblue', fontsize=8)
a.set_ylabel('độ nghiêng cẳng tay')
a.set_title(f'{TOTAL_REPS} rep / {len(SETS)} set — dải xanh = đang tập')
a.legend(loc='upper right', fontsize=8); a.grid(alpha=.25)

# (2) Góc khuỷu (đối chiếu)
ax[1].plot(t, S['angL_s'], lw=.7, label='khuỷu trái')
ax[1].plot(t, S['angR_s'], lw=.7, label='khuỷu phải', alpha=.75)
ax[1].set_ylabel('góc khuỷu (°)')
ax[1].legend(loc='upper right', fontsize=8); ax[1].grid(alpha=.25)

# (3) Chất lượng phát hiện
ax[2].fill_between(t, 0, is_working.astype(float), step='mid',
                   color='tab:green', alpha=.5, label='đang tập')
ax[2].plot(t, bconf, lw=.5, color='gray', label='box conf')
ax[2].set_ylabel('trạng thái'); ax[2].set_xlabel('thời gian (giây)')
ax[2].legend(loc='upper right', fontsize=8); ax[2].grid(alpha=.25)

plt.tight_layout(); plt.savefig(f'{OUT_DIR}/analysis.png', dpi=100); plt.show()

In [ ]:
# Phóng to set dài nhất để soi từng nhịp
if SETS:
    st = max(SETS, key=lambda s: s['n_reps'])
    m  = (t >= st['start'] - 2) & (t <= st['end'] + 2)
    plt.figure(figsize=(20, 4))
    for name, s in SIGNALS.items():
        plt.plot(t[m], s[m], lw=1.4, marker='.', ms=3, label=name)
    for r in st['reps']:
        plt.axvline(r['t_top'], color='crimson', lw=.8, alpha=.6)
    plt.axhline(REP_HI, color='crimson', ls='--', lw=.8)
    plt.axhline(REP_LO, color='steelblue', ls='--', lw=.8)
    plt.title(f"Set dài nhất: {st['n_reps']} rep, {st['dur']:.0f}s "
              f"(vạch đỏ = thời điểm đếm)")
    plt.xlabel('giây'); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()

---
## Dò ngưỡng tự động

Chạy ô này khi **đổi sang video khác**: nó lấy các mức đáy/đỉnh thực tế từ phân bố tín
hiệu rồi đề xuất `REP_HI` / `REP_LO` đặt vào 1/3 và 2/3 khoảng dao động. Chép kết quả
ngược lên ô Cấu hình rồi chạy lại từ mục *Lượt 2a*.

In [ ]:
sig_all = np.concatenate([s[np.isfinite(s)] for s in SIGNALS.values()])
lo_lvl, hi_lvl = np.percentile(sig_all, [5, 95])
span = hi_lvl - lo_lvl
print(f'Đáy (p5)  = {lo_lvl:+.2f}')
print(f'Đỉnh (p95) = {hi_lvl:+.2f}')
print(f'Biên độ    = {span:.2f}')
print('\nĐề xuất:')
print(f'  REP_LO      = {lo_lvl + span/3:+.2f}')
print(f'  REP_HI      = {lo_lvl + 2*span/3:+.2f}')
print(f'  REP_MIN_AMP = {span*0.5:.2f}')
if span < 0.5:
    print('\nCẢNH BÁO: biên độ quá nhỏ — có thể video không có nhịp cuốn rõ ràng, '
          'hoặc pose bị nhiễu. Xem lại biểu đồ trước khi tin số rep.')

---
## Xuất video có chú thích

Đọc lại video và vẽ chồng kết quả. Vì keypoint đã có sẵn từ lượt 1 nên ô này chỉ tốn
thời gian giải mã + mã hoá, không chạy lại mạng.

Đồ thị chạy được vẽ bằng `cv2.polylines`. Notebook push-up cũ dựng một `matplotlib.Figure`
**cho mỗi frame** — cực chậm, và `fig.canvas.tostring_rgb()` mà nó dùng đã bị gỡ khỏi
Matplotlib từ bản 3.10 nên ô đó giờ sẽ lỗi.

In [ ]:
def blur_face(img, kp, w, h, kp_conf=KP_CONF):
    """Làm mờ vùng mặt nếu đủ keypoint mặt tin cậy. Không có mặt trong khung -> bỏ qua."""
    pts = np.array([kp[i, :2] for i in FACE_KPS if kp[i, 2] >= kp_conf])
    if len(pts) < 2:
        return
    pad_x, pad_y = 40, 60
    x0 = max(0, int(pts[:, 0].min()) - pad_x); x1 = min(w, int(pts[:, 0].max()) + pad_x)
    y0 = max(0, int(pts[:, 1].min()) - pad_y); y1 = min(h, int(pts[:, 1].max()) + pad_y)
    roi = img[y0:y1, x0:x1]
    if roi.size == 0:
        return
    k = max(3, int(min(roi.shape[:2]) * 0.4) | 1)
    img[y0:y1, x0:x1] = cv2.GaussianBlur(roi, (k, k), 0)


def draw_skeleton(img, kp, kp_conf=KP_CONF):
    for i, j in SKELETON:
        if kp[i, 2] >= kp_conf and kp[j, 2] >= kp_conf:
            cv2.line(img, tuple(kp[i, :2].astype(int)), tuple(kp[j, :2].astype(int)),
                     (245, 245, 245), 2, cv2.LINE_AA)
    for i in range(17):
        if kp[i, 2] >= kp_conf:
            c = (0, 200, 255) if i in (L_ELB, R_ELB, L_WRI, R_WRI) else (200, 200, 200)
            cv2.circle(img, tuple(kp[i, :2].astype(int)), 5, c, -1, cv2.LINE_AA)


def draw_trace(img, k, box, n_pts=150):
    """Đồ thị tín hiệu 2 tay đang chạy, vẽ bằng cv2 (nhanh hơn matplotlib rất nhiều)."""
    x0, y0, bw, bh = box
    cv2.rectangle(img, (x0, y0), (x0 + bw, y0 + bh), (25, 25, 25), -1)
    for lvl, col in ((REP_HI, (60, 60, 235)), (REP_LO, (235, 160, 60))):
        y = int(y0 + bh * (1 - (lvl + 1) / 2))
        cv2.line(img, (x0, y), (x0 + bw, y), col, 1, cv2.LINE_AA)

    a = max(0, k - n_pts)
    for sig, col in ((S['foreL_s'], (255, 170, 60)), (S['foreR_s'], (90, 90, 255))):
        seg = sig[a:k + 1]
        if len(seg) < 2:
            continue
        xs = np.linspace(x0, x0 + bw, len(seg))
        ys = y0 + bh * (1 - (np.clip(seg, -1, 1) + 1) / 2)   # [-1,1] -> đáy..đỉnh hộp
        ok = np.isfinite(ys)
        if ok.sum() >= 2:
            pts = np.stack([xs[ok], ys[ok]], 1).astype(np.int32)
            cv2.polylines(img, [pts], False, col, 2, cv2.LINE_AA)


def draw_hud(img, k, w):
    """Bảng thông tin: tổng rep, set, trạng thái tập/nghỉ."""
    working = is_working[k]
    sid     = set_id[k]
    pw, ph  = 470, 210
    x0, y0  = w - pw - 30, 30
    panel   = img[y0:y0+ph, x0:x0+pw].copy()
    cv2.rectangle(panel, (0, 0), (pw, ph), (0, 0, 0), -1)
    img[y0:y0+ph, x0:x0+pw] = cv2.addWeighted(panel, .55, img[y0:y0+ph, x0:x0+pw], .45, 0)

    col   = (80, 230, 80) if working else (110, 110, 240)
    label = f'DANG TAP - Set {sid+1}' if working else 'DANG NGHI'
    F     = cv2.FONT_HERSHEY_SIMPLEX
    cv2.rectangle(img, (x0, y0), (x0 + pw, y0 + ph), col, 2)
    cv2.putText(img, label, (x0+18, y0+42), F, .95, col, 2, cv2.LINE_AA)
    cv2.putText(img, f'Total reps : {cum_reps[k]}', (x0+18, y0+90),
                F, .85, (255, 255, 255), 2, cv2.LINE_AA)
    if working:
        cv2.putText(img, f'Set reps   : {reps_in_set[k]}', (x0+18, y0+128),
                    F, .85, (80, 230, 80), 2, cv2.LINE_AA)
    with np.errstate(invalid='ignore'):
        ang = np.nanmean([S['angL_s'][k], S['angR_s'][k]])
    cv2.putText(img, f'Elbow  : {ang:.0f} do' if np.isfinite(ang) else 'Elbow  : --',
                (x0+18, y0+166), F, .7, (200, 200, 200), 2, cv2.LINE_AA)
    l, r = S['foreL_s'][k], S['foreR_s'][k]
    cv2.putText(img, 'L {:+.2f}   R {:+.2f}'.format(
                    l if np.isfinite(l) else float('nan'),
                    r if np.isfinite(r) else float('nan')),
                (x0+18, y0+196), F, .7, (0, 220, 255), 2, cv2.LINE_AA)
    return x0, y0 + ph + 12, pw

In [ ]:
def render(src, dst, stride, show_progress_every=2000):
    cap = cv2.VideoCapture(src)
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(dst, fourcc, FPS, (W, H))
    assert out.isOpened(), 'Không mở được VideoWriter'

    # ánh xạ chỉ số frame -> chỉ số mẫu gần nhất
    lut = np.clip(np.searchsorted(idx, np.arange(TOTAL)), 0, len(idx) - 1)

    i, t0, flash = 0, time.time(), 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        k  = lut[i] if i < len(lut) else len(idx) - 1
        kp = kpts[k]

        if np.isfinite(kp[:, 0]).any():
            if BLUR_FACE:
                blur_face(frame, kp, W, H)
            draw_skeleton(frame, kp)

        px, py, pw = draw_hud(frame, k, W)
        if DRAW_PLOT:
            draw_trace(frame, k, (px, py, pw, 150))

        # nháy viền khi vừa đếm được 1 rep
        if i > 0 and cum_reps[k] > cum_reps[lut[i-1]]:
            flash = int(FPS * 0.25)
        if flash > 0:
            cv2.rectangle(frame, (0, 0), (W-1, H-1), (0, 255, 0), 14)
            flash -= 1

        cv2.putText(frame, time.strftime('%M:%S', time.gmtime(i / FPS)),
                    (30, H - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0,
                    (255, 255, 255), 2, cv2.LINE_AA)
        out.write(frame)

        if i % show_progress_every == 0:
            el = time.time() - t0
            d  = max(i / TOTAL, 1e-9)
            print(f'  render {i}/{TOTAL} ({d*100:5.1f}%)  {el:.0f}s  '
                  f'còn ~{el/d-el:.0f}s', flush=True)
        i += 1

    cap.release(); out.release()
    print(f'Đã ghi {dst} ({os.path.getsize(dst)/1e6:.0f} MB) trong {time.time()-t0:.0f}s')


if RENDER_VIDEO:
    render(VIDEO_PATH, OUT_VIDEO, STRIDE)
else:
    print('RENDER_VIDEO = False, bỏ qua.')

### Chuyển sang H.264 để xem được trong trình duyệt

`mp4v` xuất được nhưng trình duyệt không phát. Ô dưới chuyển sang H.264 (tuỳ chọn).

In [ ]:
H264_PATH = f'{OUT_DIR}/curl_output_h264.mp4'
if RENDER_VIDEO and os.path.exists(OUT_VIDEO):
    rc = os.system(f'ffmpeg -y -loglevel error -i "{OUT_VIDEO}" '
                   f'-c:v libx264 -preset veryfast -crf 26 -pix_fmt yuv420p "{H264_PATH}"')
    print('ffmpeg rc =', rc, '|', H264_PATH if rc == 0 else 'thất bại — dùng file mp4v')

---
## Ghi chú khi chỉnh

| Triệu chứng | Chỉnh |
|---|---|
| Đếm thiếu rep (cuốn không hết tầm) | Hạ `REP_HI`, hạ `REP_MIN_AMP` |
| Đếm dư (rung tay lúc nghỉ cũng tính) | Nâng `REP_MIN_AMP`, nới `REP_HI`/`REP_LO` xa nhau hơn |
| Một nhịp bị đếm thành hai | Nâng `REP_MIN_PERIOD`, tăng `SMOOTH_WIN_S` |
| Một set bị cắt làm đôi | Nâng `SET_MAX_GAP_S` |
| Hai set dính làm một | Hạ `SET_MAX_GAP_S` |
| Tín hiệu đứt đoạn nhiều | Hạ `KP_CONF` xuống 0.35, hoặc đổi `MODEL_NAME` sang `yolo11m-pose.pt` |
| Hai tay cuốn cùng lúc bị tính thành 2 rep | Nâng `MERGE_WIN_S` |
| Cuốn xen kẽ bị gộp thành 1 rep | Hạ `MERGE_WIN_S` |
| Động tác lẻ bị tính thành set | Nâng `SET_MIN_REPS`, hạ `SET_MAX_CADENCE_S` |

Đổi bài tập khác thì chỉ cần thay tín hiệu trong `build_signals` — phần đếm rep và
tách set không phụ thuộc vào bài tập:

- **Squat**: góc gối `hông–gối–cổ chân`, hoặc chiều cao hông chuẩn hoá theo chiều dài đùi.
- **Push-up**: góc khuỷu `vai–khuỷu–cổ tay` (khi quay ngang thì vai nhìn rõ nên góc khuỷu ổn).
- **Nâng vai / vai trước**: độ nghiêng cánh tay trên `vai–khuỷu` thay cho cẳng tay.